# Local LLM Chess Agent

**METHOD SUMMARY**

The main agent is a hybrid local LLM + chess heuristic agent:
1. The current chess position is represented using FEN.
2. Legal moves are generated using python-chess.
3. A local LLM (**llama3.2**), accessed through Ollama, is prompted to choose one legal move in UCI format.
4. The chosen move is then validated against the legal move list.
5. If the LLM is unavailable or returns an invalid move, the agent falls back to a lightweight one-ply heuristic move selector.
6. The agent is evaluated against a random baseline over multiple games.
7. Statistics from those games are reported, with figures and a GIF showing gameplay.

In [ ]:
#Import packages and setup

import random
import time
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import chess
import chess.svg
import imageio.v2 as imageio
import cairosvg
from pathlib import Path
from IPython.display import display, HTML, Image

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

N_GAMES = 50
MAX_PLIES = 160      #Ensures the game does not run forever

#Local LLM Settings
USE_OLLAMA = True
OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"

# Fallback strategy

In case the LLM returns an invalid move or Ollama is not running, the agent will fall back to a simple chess heuristic.

The fallback strategy scores moves by:
1. Capturing valuable pieces
2. giving check
3. promoting pawns
4. avoiding obviously poor moves where possible

This makes the agent robust, keeping it reproducible and ensuring every move is legal.

In [ ]:
#Assign each piece a value
PIECE_VALUES = {
    chess.PAWN: 100,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.ROOK: 500,
    chess.QUEEN: 900,
    chess.KING: 0,
}

def material_score(board: chess.Board) -> int:
    """Calculated score to see who is winning, a positive score indicates White, whilst a negative
    score indicates Black has the advantage."""
    score = 0 
    for piece_type, value in PIECE_VALUES.items():
        score += len(board.pieces(piece_type, chess.WHITE)) * value
        score -= len(board.pieces(piece_type, chess.BLACK)) * value
    return score

def heuristic_move_score(board: chess.Board, move: chess.Move) -> float:
    """Score a move from the perspective of the side to move."""
    score = 0.0

    #Reward captures
    if board.is_capture(move):
        captured = board.piece_at(move.to_square)
        attacker = board.piece_at(move.from_square)
        if captured:
            score += PIECE_VALUES.get(captured.piece_type, 0)
        if attacker:
            score -= 0.1 * PIECE_VALUES.get(attacker.piece_type, 0)

    #Reward promotions
    if move.promotion:
        score += PIECE_VALUES.get(move.promotion, 0)
    
    #Reward Checks
    board.push(move)
    if board.is_check():
        score += 50
    
    #Reward material improvement after the move
    mat = material_score(board)
    score += 0.01 * (mat if board.turn == chess.BLACK else -mat)

    board.pop()

    #Add tiny noise to avoid deterministic repetition
    score += random.random() * 0.001
    return score

def choose_heuristic_move(board: chess.Board) -> chess.Move:
    legal_moves = list(board.legal_moves)
    return max(legal_moves, key=lambda move: heuristic_move_score(board, move))

# Agents

1. RandomAgent: Straightforward baseline 
2. HeuristicAgent: Fallback Agent
3. LLMChessAgent: local Ollama-based agent
4. HybridLLMChessAgent: LLM first, fallback second

In [ ]:
class RandomAgent:
    def __init__(self, name="Random Agent"):
        self.name = name

    def get_move(self, board: chess.Board) -> chess.Move:
        return random.choice(list(board.legal_moves))
    

class HeuristicAgent:
    def __init__(self, name="Heuristic Fallback Agent"):
        self.name = name

    def get_move(self, board: chess.Board) -> chess.Move:
        return choose_heuristic_move(board)
    
class LLMChessAgent:
    """
    Local LLM-based chess agent using Ollama.
    
    Model/API used:
     - Local Ollama HTTP API
     - Default model: llama3.2
     
    Legal moves are generated using python-chess. The LLM is the asked to choose from the list.
    The returned move is only accepted if it exactly matches a legal move.
    
    If the LLM returns an illegal move, the agent returns None.
    The hybrid wrapper then uses a heuristic fallback.
    """

    def __init__(
        self,
        model=OLLAMA_MODEL,
        url=OLLAMA_URL,
        temperature=0.1,
        timeout=20,
        name=None,
    ):
        self.model = model
        self.url = url
        self.temperature = temperature
        self.timeout = timeout
        self.name = name or f"Local LLM Chess Agent ({model})"
        self.invalid_count = 0
        self.valid_count = 0
        self.error_count = 0
        self.raw_outputs = []
    
    def build_prompt(self, board: chess.Board) -> str:
        board_fen = board.fen()
        legal_moves = [move.uci() for move in board.legal_moves]
        side = "White" if board.turn == chess.WHITE else "Black"

        prompt = f"""
You are a strong chess-playing agent.

Current board in FEN:
{board_fen}

Side to move:
{side}

Legal moves in UCI format:
{', '.join(legal_moves)}

Choose the legal move that gives {side} the best chance of winning.

Prioritise, in order:
1. Checkmate, if available
2. Avoid being checkmated
3. Capturing high-value pieces safely
4. Giving check when useful
5. Promoting pawns
6. Improving piece activity and king safety

Rules:
1. Only return one move in UCI format.
2. The move must be exactly one of the legal moves listed above.
3. Do not explain.
4. Do not include punctuation.
"""
        return prompt.strip()
    
    def call_ollama(self, prompt: str) -> str:
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options":{
                "temperature": self.temperature,
                "num_predict": 12,
            },
        }
        response = requests.post(self.url, json=payload, timeout=self.timeout)
        response.raise_for_status()
        data = response.json()
        return data.get("response", "").strip()
    
    def extract_uci_move(self, text: str, legal_moves):
        cleaned = text.strip().replace("`", "").replace(".", "").replace(",","")
        tokens = cleaned.split()

        if cleaned in legal_moves:
            return cleaned
        
        for token in tokens:
            if token in legal_moves:
                return token
            
        return None
    
    def get_move(self, board: chess.Board):
        legal_moves = [move.uci() for move in board.legal_moves]
        prompt = self.build_prompt(board)

        try:
            raw = self.call_ollama(prompt)
            self.raw_outputs.append(raw)
            move_uci = self.extract_uci_move(raw, legal_moves)

            if move_uci in legal_moves:
                self.valid_count += 1
                return chess.Move.from_uci(move_uci)
            
            self.invalid_count += 1
            return None
        
        except Exception as exc:
            self.error_count += 1
            self.raw_outputs.append(f"ERROR: {exc}")
            return None
        
class HybridLLMChessAgent:
    def __init__(self, use_llm=USE_OLLAMA, model=OLLAMA_MODEL):
        self.use_llm = use_llm
        self.llm_agent = LLMChessAgent(model=model)
        self.fallback_agent = HeuristicAgent()
        self.name = f"Hybrid Local LLM + Heuristic ({model})" if use_llm else "Heuristic Agent (LLM disabled)"

    def get_move(self, board: chess.Board) -> chess.Move:
        if self.use_llm:
            move = self.llm_agent.get_move(board)
            if move in board.legal_moves:
                return move
            
        return self.fallback_agent.get_move(board)


# Game Playing and Evaluation

The evaluation alternates colours so that the tested agents plays as both White and Black.

In [ ]:
def play_game(agent_white, agent_black, max_plies=MAX_PLIES, record_positions=False):
    board = chess.Board()
    positions = [board.copy()] if record_positions else None

    for ply in range(max_plies):
        if board.is_game_over(claim_draw=True):
            break

        agent = agent_white if board.turn == chess.WHITE else agent_black
        move = agent.get_move(board)

        if move not in board.legal_moves:
            move = random.choice(list(board.legal_moves))

        board.push(move)

        if record_positions:
            positions.append(board.copy())

    result = board.result(claim_draw=True) if board.is_game_over(claim_draw=True) else "1/2-1/2"
    return board, result, positions

def result_from_agent_perspective(result, agent_colour):
    if result == "1/2-1/2":
        return "draw"
    if result == "1-0":
        return "win" if agent_colour == chess.WHITE else "loss"
    if result == "0-1":
        return "win" if agent_colour == chess.BLACK else "loss"
    return "draw"


def run_evaluation(agent, opponent, rounds = 50, max_plies=MAX_PLIES):
    stats = {
        "wins": 0,
        "losses": 0,
        "draws": 0,
        "history": []
    }

    for game_idx in range(rounds):
        agent_colour = chess.WHITE if game_idx % 2 == 0 else chess.BLACK

        if agent_colour == chess.WHITE:
            board, result, _ = play_game(agent, opponent, max_plies=max_plies)
        else:
            board, result, _ = play_game(opponent, agent, max_plies=max_plies)

        outcome = result_from_agent_perspective(result, agent_colour)
        if outcome == "win":
            stats["wins"] += 1
        elif outcome == "loss":
            stats["losses"] += 1
        else:
            stats["draws"] += 1

        stats["history"].append({
            "game": game_idx + 1,
            "agent_colour": "White" if agent_colour == chess.WHITE else "Black",
            "result": result,
            "agent_outcome": outcome,
            "plies": board.ply(),
            "termination": board.outcome(claim_draw=True).termination.name if board.outcome(claim_draw=True) else "MAX_PLIES_DRAW",
        })
    
    return stats

def stats_to_dataframe(stats, agent_name, opponent_name):
    n = len(stats["history"])
    summary = pd.DataFrame([
        {
            "Agent": agent_name,
            "Opponent": opponent_name,
            "Games": n,
            "Wins": stats["wins"],
            "Draws": stats["draws"],
            "Losses": stats["losses"],
            "Win Rate": stats["wins"] / n,
            "Draw Rate": stats["draws"] / n,
            "Loss Rate": stats["losses"] / n
        }
    ])
    history = pd.DataFrame(stats["history"])
    return summary, history


# Run Evaluation against the Random Baseline

In [ ]:
agent = HybridLLMChessAgent(use_llm=USE_OLLAMA, model=OLLAMA_MODEL)
baseline = RandomAgent()

print("Evaluating:", agent.name, "vs", baseline.name)
stats = run_evaluation(agent, baseline, rounds=N_GAMES)

summary_df, history_df = stats_to_dataframe(stats, agent.name, baseline.name)

display(summary_df)
display(history_df.head(10))

In [ ]:
plot_df = summary_df[["Wins", "Draws", "Losses"]].T
plot_df.columns = ["Count"]

ax = plot_df.plot(kind="bar", legend=False, figsize=(7, 4))
ax.set_title(f"{agent.name} vs Random Baseline over {N_GAMES} games")
ax.set_xlabel("Outcome")
ax.set_ylabel("Number of games")

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
rate_df = summary_df[["Win Rate", "Draw Rate", "Loss Rate"]].T
rate_df.columns = ["Rate"]

ax = rate_df.plot(kind="bar", legend=False, figsize=(7, 4))
ax.set_title("Win/Draw/Loss Rates")
ax.set_xlabel("Outcome")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Demo of game

In [ ]:
demo_agent = HybridLLMChessAgent(use_llm=USE_OLLAMA, model=OLLAMA_MODEL)
demo_baseline = RandomAgent()

demo_board, demo_result, demo_positions = play_game(
    demo_agent,
    demo_baseline,
    max_plies=80,
    record_positions=True,
)

print("Demo result:", demo_result)
print("Number of plies:", demo_board.ply())
display(HTML(chess.svg.board(demo_board, size=420)))

# GIF

In [ ]:
output_dir = Path("chess_demo_frames")
output_dir.mkdir(exist_ok=True)

frame_paths = []
for i, pos in enumerate(demo_positions):
    svg = chess.svg.board(pos, size=420)
    png_path = output_dir / f"frame_{i:03d}.png"
    cairosvg.svg2png(bytestring=svg.encode("utf-8"), write_to=str(png_path))
    frame_paths.append(png_path)

gif_path = Path("chess_agent_demo.gif")
frames = [imageio.imread(path) for path in frame_paths]
imageio.mimsave(gif_path, frames, duration=0.45)

print("Saved GIF to:", gif_path.resolve())
display(Image(filename=str(gif_path)))

# Final Checks to make sure LLM worked

In [ ]:

if hasattr(agent, "llm_agent"):
    print("LLM valid moves:", agent.llm_agent.valid_count)
    print("LLM invalid moves:", agent.llm_agent.invalid_count)
    print("LLM errors:", agent.llm_agent.error_count)
    print("Recent raw outputs:")
    for out in agent.llm_agent.raw_outputs[-5:]:
        print("-", out)